# PACT — 01: run the experiments

**Fidelity Is a Certificate: Distilling Black-Box Teachers into Analytically
Decomposable PCE–HDMR Students**

This notebook runs the full protocol from the report:

| | |
|---|---|
| datasets | 4 real OpenML tabular sets (`bank-marketing`, `credit-default`, `gas-drift`, `musk v2`) |
| seeds | 10 |
| teacher | MLP, **50 epochs** (plus a GBM ablation) |
| students | **GAS-PCE** (ours), soft decision tree, GA²M |
| reference | Monte-Carlo Sobol' on the teacher + KernelSHAP + permutation importance |

Each `(dataset, teacher, ρ, seed)` cell is written to its own JSON file, so the
run is **resumable**: if the Colab session dies, re-running this notebook skips
everything already on disk.

> **Runtime.** The full grid is roughly 2–4 h on a T4. Start with
> `QUICK = True` (a few minutes) to confirm everything works, then switch it
> off. Save results to Drive so a disconnect does not cost you the run.

In [ ]:
#@title Setup — clone the repo and install dependencies { display-mode: "form" }
REPO_URL  = "https://github.com/<user>/pact-hdmr-kd.git"  #@param {type:"string"}
USE_DRIVE = True   #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/pact_runs"  #@param {type:"string"}

import os, subprocess, sys, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not pathlib.Path("/content/pact-hdmr-kd").exists():
        # If you uploaded the repo as a zip instead, unzip it to /content and
        # the clone is skipped automatically.
        try:
            subprocess.run(["git", "clone", "-q", REPO_URL,
                            "/content/pact-hdmr-kd"], check=True)
        except Exception as e:
            print("clone failed:", e)
            print("Upload the repo zip to /content and unzip it, then re-run.")
    os.chdir("/content/pact-hdmr-kd")

    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "interpret-core==0.6.*", "shap>=0.44", "openml"], check=False)

    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        pathlib.Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, os.getcwd())

import numpy as np, torch
print("cwd       :", os.getcwd())
print("torch     :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu       :", torch.cuda.get_device_name(0))

## 1. Configuration

Everything a reviewer would want to check lives in `pact/config.py`. Nothing in
this notebook hard-codes a hyperparameter.

In [ ]:
from pact.config import RunConfig, DATASETS, RHO_ABLATION, TEACHER_ABLATION

QUICK = True            #@param {type:"boolean"}  ← set False for the real run
OUT_DIR = DRIVE_DIR if (IN_COLAB and USE_DRIVE) else "runs"

cfg = RunConfig()
if QUICK:
    cfg = cfg.to_quick()
cfg.out_dir = OUT_DIR

print(f"output dir : {OUT_DIR}")
print(f"seeds      : {list(cfg.seeds)}")
print(f"epochs     : {cfg.epochs}")
print(f"PCE        : p={cfg.degree}, q={cfg.q_norm}, m={cfg.screen_m}, "
      f"gamma={cfg.gamma}, |lambda path|={cfg.n_lambda}")
print()
for d in DATASETS:
    print(f"  {d.key:7s} OpenML {d.openml_id:<6d} {d.regime:9s} {d.note}")

## 2. Sanity check on one cell

Run a single `(bank, seed 0)` cell first. If this works, the grid will too.
The one number to watch is `holds=True` in the certificate line — Theorem 1 is
not an empirical claim, so a `False` here means a bug in the implementation,
not an interesting result.

In [ ]:
from pact.experiment import run_cell

rec = run_cell("bank", seed=0, cfg=cfg, rho=cfg.rho, teacher_kind="mlp",
               out_dir=OUT_DIR)

g = rec["students"]["gas_pce"]
print()
print(f"teacher test acc : {rec['teacher']['acc_test']:.4f}")
print(f"eta_mu / eta_nu  : {g['fidelity_mu']['eta']:.4f} / "
      f"{g['fidelity_nu']['eta']:.4f}")
print(f"top-1 agreement  : {g['fidelity_mu']['top1_agreement']:.4f}")
print(f"components       : {g['complexity']['active_components']} "
      f"(basis {g['complexity']['basis_size']})")
c = g["certificate"]
print(f"certificate      : Delta={c['bound_delta']:.4f}  "
      f"measured={c['measured_l1']:.4f}  holds={c['bound_holds']}")
print(f"vs teacher Sobol : Spearman {g['agreement_vs_mc_sobol']['spearman']:.3f}")

## 3. Main grid — 4 datasets × 10 seeds

This is the run behind Tables 2–4. Already-completed cells are skipped.

In [ ]:
from pact.experiment import run_grid

main_runs = run_grid(cfg,
                     dataset_keys=[d.key for d in DATASETS],
                     rhos=[cfg.rho],
                     teachers=["mlp"],
                     out_dir=OUT_DIR)
print(f"\ncompleted cells in this pass: {len(main_runs)}")

## 4. Ablation over ρ (the product-measure query fraction)

This is the experiment behind Proposition 3: at ρ = 0 the student is fitted
only on the data manifold, so its certificate should be weak even when its
on-manifold fidelity looks excellent. The ablation runs on the two moderate
datasets to keep the cost down; set `ABLATE_ALL = True` for the full version
(four datasets x three rho values x ten seeds = 120 cells, so stage it across
sessions -- `run_grid` resumes).

Note that rho = `cfg.rho` is excluded here because the main grid already ran
it. That is only sound if those cells came from the same package version as
this pass; `load_runs` checks and will tell you if the rho axis ends up
partitioned across code versions.

In [ ]:
ABLATE_ALL = False   #@param {type:"boolean"}
REDUCED_KEYS = ["bank", "credit"]      # the cheap subset when ABLATE_ALL=False

ablation_keys = [d.key for d in DATASETS] if ABLATE_ALL else REDUCED_KEYS
ablation_rhos = [r for r in RHO_ABLATION if abs(r - cfg.rho) > 1e-9]

# State the plan before spending the cells. This is the most expensive stage
# in the notebook and it is normally run across several Colab sessions, so a
# surprise here costs hours. run_grid resumes, so re-running this cell after a
# disconnect only does what is missing.
if ABLATE_ALL and sorted(ablation_keys) == sorted(REDUCED_KEYS):
    print("[note] ABLATE_ALL has no effect: REDUCED_KEYS already lists every "
          "dataset. Shorten REDUCED_KEYS if you want the toggle to reduce cost.")
planned = len(ablation_keys) * len(ablation_rhos) * len(cfg.seeds)
print(f"plan: {ablation_keys} x rho={ablation_rhos} x {len(cfg.seeds)} seeds "
      f"x teacher=mlp  ->  {planned} cells")
print(f"      rho={cfg.rho:g} is NOT re-run here; it comes from the main grid "
      f"above, so those cells must have been produced by the SAME package "
      f"version or the rho column of the ablation table compares code "
      f"versions as well as rho (load_runs will say so).")

rho_runs = run_grid(cfg,
                    dataset_keys=ablation_keys,
                    rhos=ablation_rhos,
                    teachers=["mlp"],
                    out_dir=OUT_DIR)
print(f"\nrho-ablation cells this pass: {len(rho_runs)}")

## 5. Ablation over the teacher family

Gradient-boosted trees still tend to beat deep models on tabular data
(Borisov et al., 2024), so the teacher is an experimental variable rather than
a given. If the certificate behaves differently under a GBM teacher, that is
worth knowing.

In [ ]:
gbm_runs = run_grid(cfg,
                    dataset_keys=["bank", "credit"],
                    rhos=[cfg.rho],
                    teachers=["gbm"],
                    out_dir=OUT_DIR)
print(f"\nGBM-teacher cells this pass: {len(gbm_runs)}")

## 6. Inventory

A short audit of what is actually on disk, so you know exactly what the tables
in notebook 02 will be built from.

In [ ]:
from collections import Counter
from pact.experiment import load_runs

runs = load_runs(OUT_DIR)
print(f"total run records: {len(runs)}\n")

by = Counter((r["dataset"], r.get("teacher_kind", "mlp"), r["rho"]) for r in runs)
for (d, t, rho), n in sorted(by.items()):
    print(f"  {d:7s} teacher={t:4s} rho={rho:<5g} seeds={n}")

# bound_holds is None where eta_mu >= 1: Theorem 1 assumes eta < 1, so there
# is no claim to test and the cell is NOT a violation. Counting `not None` as a
# failure would report the paper's headline Prop. 3 finding as a bug.
_certs = [(r["cell"], r["students"]["gas_pce"]["certificate"]) for r in runs]
bad  = [c for c, k in _certs if k.get("bound_holds") is False]
viol = [c for c, k in _certs if k.get("precondition_violated")]
print(f"\ncertificate violations: {len(bad)}   "
      f"(cells with eta_mu >= 1, not tested: {len(viol)})")
if viol:
    print("  eta_mu >= 1 (expected at rho=0, Prop. 3):", viol[:5])
if bad:
    print("  ", bad[:10])
    print("  ^ this is a BUG, not a result — open an issue before writing it up")

---
Next: **`02_PACT_analysis_figures_tables.ipynb`** turns these JSON records into
the LaTeX tables and PDF figures that `paper/main.tex` inputs.